# A2C (Advantage Actor-Critic): training notebook

Built from [`training/a2c_training.py`](../training/a2c_training.py). It uses
**TorchRL** (environment wrapper, data collector, loss module) to train the blood
delivery drone.

A2C is an actor-critic method. The **actor** picks actions and the **critic** estimates how good a state is. The **advantage** (reward compared with what the critic expected) tells the actor how much better an action was than usual. Each batch gives one gradient step.

**Contents:** 1. setup, 2. hyperparameters, 3. the training function, 4. train,
5. learning curves, 6. test the trained agent, 7. hyperparameter experiments.

## 1. Setup

In [ ]:
import os
import sys

# Run from the project root so `environment/`, `training/` and `logs/` are found.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

print("Working folder:", os.getcwd())

### Imports

TorchRL provides the environment wrapper, the collector and the loss module.

In [ ]:
import random
import time
from pathlib import Path

import numpy as np
import torch
from environment.custom_env import BloodDroneEnv
from tensordict.nn import TensorDictModule
from torch import nn
from torch.distributions import Categorical
from torchrl.collectors import Collector
from torchrl.envs import Compose, GymWrapper, RewardScaling, RewardSum, TransformedEnv
from torchrl.modules import MLP, ProbabilisticActor, ValueOperator
from torchrl.objectives import A2CLoss
from torchrl.objectives.value import GAE

### Helpers

Small functions for the environment, the network, evaluation and logging. The same code is at the top of all four training files.

In [ ]:
ENV_SETTINGS = {"grid_size": 10, "max_steps": 200}


COMMON_DEFAULTS = {
    "hidden": (64, 64),         # two hidden layers of 64 units, ReLU
    "total_frames": 100_000,    # environment steps used for training
    "seed": 0,
    "eval_every": 5_000,        # frames between evaluation checkpoints
    "eval_episodes": 10,        # episodes per checkpoint
    "final_eval_episodes": 50,  # episodes for the final evaluation
}


CHECKPOINT_SEED = 500


FINAL_EVAL_SEED = 1000


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_env(seed=None, reward_scale=1.0):
    """
    The drone environment as a TorchRL environment.

    reward_scale shrinks the reward the learner sees (the reported episode
    reward is always the real one). Policy-gradient methods learn better when
    rewards are small numbers.
    """

    transforms = [RewardSum()]
    if reward_scale != 1.0:
        transforms.append(RewardScaling(loc=0.0, scale=reward_scale))

    env = TransformedEnv(
        GymWrapper(
            BloodDroneEnv(**ENV_SETTINGS),
            categorical_action_encoding=True,
        ),
        Compose(*transforms),
    )
    if seed is not None:
        env.set_seed(seed)
    return env


def env_sizes():
    """(observation size, number of actions)."""

    env = BloodDroneEnv(**ENV_SETTINGS)
    return env.observation_space.shape[0], env.action_space.n


def make_mlp(in_features, out_features, hidden):
    """Fully connected network with ReLU activations."""

    return MLP(
        in_features=in_features,
        out_features=out_features,
        num_cells=list(hidden),
        activation_class=nn.ReLU,
    )


def finished_episode_rewards(batch):
    """Total rewards of the episodes that ended inside this batch."""

    done = batch["next", "done"].reshape(-1)
    rewards = batch["next", "episode_reward"].reshape(-1)
    return rewards[done].tolist()


def greedy_action(network):
    """Function that picks the best action for an observation."""

    def act(observation):
        with torch.no_grad():
            scores = network(torch.as_tensor(observation, dtype=torch.float32))
            return int(scores.argmax().item())

    return act


def evaluate(act, episodes=10, seed_start=FINAL_EVAL_SEED):
    """
    Fly complete missions and measure how well the agent does.

    Returns the metrics used in the report: average reward, delivery success,
    steps per delivery, battery and package integrity at delivery, tower hits.
    """

    env = BloodDroneEnv(**ENV_SETTINGS)
    rewards, delivered, complete, steps, hits = [], [], [], [], []
    delivery_steps, delivery_battery, delivery_integrity = [], [], []

    for k in range(episodes):
        observation, _ = env.reset(seed=seed_start + k)
        total = 0.0
        done = False
        was_delivered = False

        while not done:
            observation, reward, terminated, truncated, _ = env.step(act(observation))
            total += reward
            done = terminated or truncated

            if env.delivered and not was_delivered:
                was_delivered = True
                delivery_steps.append(env.steps)
                delivery_battery.append(env.battery)
                delivery_integrity.append(env.package_integrity)

        rewards.append(total)
        delivered.append(float(env.delivered))
        complete.append(float(env.mission_complete))
        steps.append(env.steps)
        hits.append(env.obstacle_hits)

    def mean(values):
        return float(np.mean(values)) if len(values) else None

    return {
        "episodes": episodes,
        "avg_reward": mean(rewards),
        "std_reward": float(np.std(rewards)),
        "delivery_success": mean(delivered),
        "return_rate": mean(complete),
        "avg_steps": mean(steps),
        "avg_delivery_steps": mean(delivery_steps),
        "avg_battery_at_delivery": mean(delivery_battery),
        "avg_integrity_at_delivery": mean(delivery_integrity),
        "avg_tower_hits": mean(hits),
    }


def save_checkpoint(path, algorithm, network, hidden):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "algorithm": algorithm,
            "hidden": list(hidden),
            "state_dict": network.state_dict(),
        },
        path,
    )


class RunLog:
    """Collects the learning curves and results of one training run."""

    def __init__(self, algorithm, name, config):
        self.algorithm = algorithm
        self.name = name
        self.config = {k: (list(v) if isinstance(v, tuple) else v) for k, v in config.items()}
        self.history = []     # one entry per collected batch
        self.evals = []       # one entry per evaluation checkpoint
        self.start = time.time()
        self.next_eval = config["eval_every"]

    def add(self, **values):
        self.history.append({**values, "time": time.time() - self.start})

    def due_for_eval(self, frames):
        if frames >= self.next_eval:
            self.next_eval += self.config["eval_every"]
            return True
        return False

    def add_eval(self, frames, metrics):
        self.evals.append({"frames": frames, **metrics})

    def stability(self):
        """Standard deviation of training reward over the last 20% of training."""

        rewards = [h["train_reward"] for h in self.history if h.get("train_reward") is not None]
        tail = rewards[int(len(rewards) * 0.8):] or rewards
        return float(np.std(tail)) if tail else None


def log_line(verbose, algorithm, frames, total, train_reward, extra=""):
    if verbose:
        reward = "n/a" if train_reward is None else f"{train_reward:8.1f}"
        print(f"[{algorithm}] frames {frames:>7}/{total} | train reward {reward} {extra}")

## 2. Hyperparameters

Default values used by every run (`DEFAULTS`). The sweep changes these one at a
time: lr, gamma, entropy_coeff, value_coeff, frames_per_batch, gae_lambda. All algorithms share the same network (two hidden layers
of 64 units, ReLU), the same environment and the same training budget.

In [ ]:
DEFAULTS = {
    **COMMON_DEFAULTS,
    "lr": 3e-4,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "entropy_coeff": 0.01,
    "value_coeff": 0.5,
    "grad_clip": 1.0,
    "reward_scale": 1.0,       # the learner sees reward * reward_scale
    "frames_per_batch": 200,   # environment steps per policy update
}

## 3. Training function

`train_a2c` collects experience with TorchRL, updates the network, evaluates it every 5,000 steps and saves a log.

In [ ]:
def train_a2c(config=None, run_name="a2c_default", model_path=None,
              verbose=True):
    """Train the drone with A2C and return the final evaluation."""

    cfg = {**DEFAULTS, **(config or {})}
    seed_everything(cfg["seed"])

    probe = make_env(reward_scale=cfg["reward_scale"])
    obs_size, n_actions = env_sizes()

    # Actor: observation -> action probabilities. Critic: observation -> value.
    actor_network = make_mlp(obs_size, n_actions, cfg["hidden"])
    actor = ProbabilisticActor(
        TensorDictModule(actor_network, in_keys=["observation"], out_keys=["logits"]),
        in_keys=["logits"],
        spec=probe.action_spec,
        distribution_class=Categorical,
        return_log_prob=True,
    )
    critic = ValueOperator(
        make_mlp(obs_size, 1, cfg["hidden"]), in_keys=["observation"]
    )

    advantage = GAE(
        gamma=cfg["gamma"], lmbda=cfg["gae_lambda"], value_network=critic
    )
    loss_module = A2CLoss(
        actor,
        critic,
        entropy_bonus=True,
        entropy_coeff=cfg["entropy_coeff"],
        critic_coeff=cfg["value_coeff"],
    )
    optimizer = torch.optim.Adam(loss_module.parameters(), lr=cfg["lr"])

    collector = Collector(
        lambda: make_env(cfg["seed"], cfg["reward_scale"]),
        actor,
        frames_per_batch=cfg["frames_per_batch"],
        total_frames=cfg["total_frames"],
        device="cpu",
    )

    log = RunLog("a2c", run_name, cfg)
    frames = 0
    episodes = 0

    for batch in collector:
        frames += batch.numel()
        episode_rewards = finished_episode_rewards(batch)
        episodes += len(episode_rewards)

        with torch.no_grad():
            advantage(batch)

        loss_td = loss_module(batch)
        loss = (
            loss_td["loss_objective"]
            + loss_td["loss_critic"]
            + loss_td["loss_entropy"]
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(loss_module.parameters(), cfg["grad_clip"])
        optimizer.step()

        train_reward = (
            sum(episode_rewards) / len(episode_rewards) if episode_rewards else None
        )
        entropy_value = loss_td["entropy"].mean().item()
        log.add(
            frames=frames,
            episodes=episodes,
            train_reward=train_reward,
            loss=loss.item(),
            entropy=entropy_value,
        )

        if log.due_for_eval(frames):
            metrics = evaluate(greedy_action(actor_network), cfg["eval_episodes"], CHECKPOINT_SEED)
            log.add_eval(frames, metrics)
            log_line(
                verbose, "A2C", frames, cfg["total_frames"], train_reward,
                f"| eval reward {metrics['avg_reward']:7.1f} | "
                f"success {metrics['delivery_success']:.0%} | entropy {entropy_value:.2f}",
            )

    collector.shutdown()

    final = evaluate(greedy_action(actor_network), cfg["final_eval_episodes"], FINAL_EVAL_SEED)
    if model_path:
        save_checkpoint(model_path, "a2c", actor_network, cfg["hidden"])

    if verbose:
        print(f"\nA2C finished: reward {final['avg_reward']:.1f}, "
              f"delivery success {final['delivery_success']:.0%}")
    return {"final": final, "log": log, "network": actor_network}

## 4. Train

`TOTAL_STEPS` is the number of environment steps. The default 100,000 takes a few
minutes. This cell does **not** overwrite the saved model in `models/a2c/`.

In [ ]:
TOTAL_STEPS = 100_000

result = train_a2c(
    {"total_frames": TOTAL_STEPS},
    run_name="notebook_a2c",
)

## 5. Learning curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

history = result["log"].history
evals = result["log"].evals

frames = [h["frames"] for h in history if h.get("train_reward") is not None]
reward = [h["train_reward"] for h in history if h.get("train_reward") is not None]
frames_l = [h['frames'] for h in history if h.get('entropy') is not None]
entropy = [h['entropy'] for h in history if h.get('entropy') is not None]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

window = 10
axes[0].plot(frames, reward, alpha=0.3, label="per iteration")
if len(reward) >= window:
    axes[0].plot(frames[window - 1:], np.convolve(reward, np.ones(window) / window, mode="valid"), label="moving average")
axes[0].axhline(0, color="grey", linewidth=0.8)
axes[0].set_title("Training episode reward")
axes[0].legend()

axes[1].plot(frames_l, entropy)
axes[1].set_title('Policy entropy')

axes[2].plot([e["frames"] for e in evals], [e["avg_reward"] for e in evals], marker="o")
axes[2].set_title("Evaluation reward (10 episodes)")

for ax in axes:
    ax.set_xlabel("Environment steps")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Test the trained agent

100 episodes with weather the agent never saw during training.

In [ ]:
HELD_OUT_SEED = 5000   # episodes the agent never saw in training

metrics = evaluate(greedy_action(result["network"]), 100, HELD_OUT_SEED)
for key, value in metrics.items():
    print(f"{key:<28} {value}")

## 7. Hyperparameter experiments

Ten A2C experiments on the **same environment**, with the same training
budget, seed and evaluation missions. Experiment 1 is the baseline (the default
values) and every other experiment changes **one** hyperparameter, so any
difference in the results comes from that one change.

Hyperparameters tested: learning rate, gamma, value coefficient, entropy coefficient and hidden layer size.

### 7.1 The 10 experiments

In [ ]:
a2c_configs = [
    {"name": "A2C-1", "lr": 0.0003, "gamma": 0.99, "value_coeff": 0.5, "entropy_coeff": 0.01, "hidden": (64, 64)},
    {"name": "A2C-2", "lr": 0.001, "gamma": 0.99, "value_coeff": 0.5, "entropy_coeff": 0.01, "hidden": (64, 64)},
    {"name": "A2C-3", "lr": 0.0001, "gamma": 0.99, "value_coeff": 0.5, "entropy_coeff": 0.01, "hidden": (64, 64)},
    {"name": "A2C-4", "lr": 0.0003, "gamma": 0.95, "value_coeff": 0.5, "entropy_coeff": 0.01, "hidden": (64, 64)},
    {"name": "A2C-5", "lr": 0.0003, "gamma": 0.999, "value_coeff": 0.5, "entropy_coeff": 0.01, "hidden": (64, 64)},
    {"name": "A2C-6", "lr": 0.0003, "gamma": 0.99, "value_coeff": 0.25, "entropy_coeff": 0.01, "hidden": (64, 64)},
    {"name": "A2C-7", "lr": 0.0003, "gamma": 0.99, "value_coeff": 0.75, "entropy_coeff": 0.01, "hidden": (64, 64)},
    {"name": "A2C-8", "lr": 0.0003, "gamma": 0.99, "value_coeff": 0.5, "entropy_coeff": 0.001, "hidden": (64, 64)},
    {"name": "A2C-9", "lr": 0.0003, "gamma": 0.99, "value_coeff": 0.5, "entropy_coeff": 0.02, "hidden": (64, 64)},
    {"name": "A2C-10", "lr": 0.0003, "gamma": 0.99, "value_coeff": 0.5, "entropy_coeff": 0.01, "hidden": (128, 128)},
]

print(f"OK: {len(a2c_configs)} A2C configurations defined")

### 7.2 Run the experiments

`TOTAL_FRAMES` is the number of environment steps for each experiment. This runs
the 10 experiments one after another and does **not** change the saved model in
`models/a2c/`.

In [ ]:
TOTAL_FRAMES = 100_000   # environment steps per experiment (the same for all 10)

# Finished experiments stay in `results`, so running this cell again continues
# where it stopped. Delete `results` (or restart the kernel) to start over.
results = globals().get("results", {})

for experiment in a2c_configs:
    name = experiment["name"]
    if name in results:
        continue

    config = {k: v for k, v in experiment.items() if k != "name"}
    print(f"Running {name}: {config}")
    results[name] = train_a2c({**config, "total_frames": TOTAL_FRAMES}, run_name=name, verbose=False)

    final = results[name]["final"]
    print(f"  {name}: reward {final['avg_reward']:.1f}, delivery success {final['delivery_success']:.0%}")

### 7.3 Results table

Final evaluation of every experiment (50 missions with weather the agent never
saw in training). `loss` and `entropy` are averages over the last 10% of training.

In [ ]:
def tail_mean(history, key, share=0.1):
    values = [h[key] for h in history if h.get(key) is not None]
    if not values:
        return None
    return sum(values[-max(1, int(len(values) * share)):]) / max(1, int(len(values) * share))


def changed(experiment):
    base = a2c_configs[0]
    diff = [f"{k} = {v}" for k, v in experiment.items() if k != "name" and base.get(k) != v]
    return ", ".join(diff) if diff else "baseline"


rows = []
for experiment in a2c_configs:
    run = results.get(experiment["name"])
    if run is None:
        continue
    final = run["final"]
    history = run["log"].history
    rows.append((
        experiment["name"], changed(experiment), final["avg_reward"],
        100 * final["delivery_success"], 100 * final["return_rate"],
        tail_mean(history, "loss"), tail_mean(history, 'entropy'),
    ))

best_reward = max(r[2] for r in rows) if rows else None
print(f"{'experiment':<14}{'changed':<28}{'reward':>9}{'deliver%':>10}{'return%':>9}{'loss':>10}{'entropy':>9}")
for name, change, reward, deliver, back, loss, entropy in rows:
    star = "  <- best" if reward == best_reward else ""
    loss_text = "" if loss is None else f"{loss:.3f}"
    entropy_text = "" if entropy is None else f"{entropy:.3f}"
    print(f"{name:<14}{change:<28}{reward:>9.1f}{deliver:>10.1f}{back:>9.1f}{loss_text:>10}{entropy_text:>9}{star}")

### 7.4 Compare the experiments

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

names = [e["name"] for e in a2c_configs if e["name"] in results]
colors = plt.cm.tab10(np.linspace(0, 1, 10))


def smooth(values, window=10):
    values = np.asarray(values, dtype=float)
    return values if len(values) < window else np.convolve(values, np.ones(window) / window, mode="valid")


fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for color, name in zip(colors, names):
    log = results[name]["log"]
    axes[0].plot([e["frames"] for e in log.evals], [e["avg_reward"] for e in log.evals], color=color, label=name)

    pairs = [(h["frames"], h["loss"]) for h in log.history if h.get("loss") is not None]
    if pairs:
        xs, ys = zip(*pairs)
        ys = smooth(ys)
        axes[1].plot(xs[len(xs) - len(ys):], ys, color=color, label=name)

    pairs = [(h["frames"], h["entropy"]) for h in log.history if h.get("entropy") is not None]
    if pairs:
        xs, ys = zip(*pairs)
        ys = smooth(ys)
        axes[2].plot(xs[len(xs) - len(ys):], ys, color=color, label=name)

axes[2].set_title("Policy entropy (smoothed)")

axes[0].set_title("Evaluation reward during training")
axes[0].legend(fontsize=7, ncol=2)
axes[1].set_title("Training loss (smoothed)")
for ax in axes:
    ax.set_xlabel("Environment steps")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Final reward of every experiment (best in green).
rewards = [results[n]["final"]["avg_reward"] for n in names]
best = int(np.argmax(rewards))
plt.figure(figsize=(9, 3.5))
bars = plt.bar(names, rewards, color=["#2f9e44" if i == best else "#4c6ef5" for i in range(len(names))])
plt.bar_label(bars, fmt="%.0f", fontsize=8)
plt.axhline(0, color="grey", linewidth=0.8)
plt.title("A2C: final average reward of each experiment")
plt.ylabel("Average reward")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### 7.5 Best hyperparameters

The best experiment has the highest average reward. If two are tied, the higher
delivery success wins, then the smaller final loss. Set `APPLY_BEST = True` to
write these values into `DEFAULTS` in `training/a2c_training.py`, so every
normal training run uses them.

In [ ]:
APPLY_BEST = False   # set to True to write the best values into training/a2c_training.py

def final_loss(name):
    value = tail_mean(results[name]["log"].history, "loss")
    return 0.0 if value is None else value

# Highest reward wins; then higher delivery success; then smaller loss.
best_name = max(
    results,
    key=lambda n: (results[n]["final"]["avg_reward"], results[n]["final"]["delivery_success"], -final_loss(n)),
)
best_experiment = next(e for e in a2c_configs if e["name"] == best_name)
best_config = {k: v for k, v in best_experiment.items() if k != "name"}

print(f"Best A2C experiment: {best_name} (average reward {results[best_name]['final']['avg_reward']:.1f})")
for key, value in best_config.items():
    print(f"  {key:<16} {value}")

if APPLY_BEST:
    import re

    path = "training/a2c_training.py"
    source = open(path, encoding="utf-8").read()
    start = source.index("\nDEFAULTS = {")
    end = source.index("\n}", start)
    block = source[start:end]

    for key, value in best_config.items():
        pattern = re.compile(rf'(\n\s*"{key}":\s*)([^,#\n]+)(,)')
        match = pattern.search(block)
        if match:
            block = block[:match.start(2)] + repr(value) + block[match.end(2):]
        else:
            block += f'\n    "{key}": {value!r},'

    open(path, "w", encoding="utf-8").write(source[:start] + block + source[end:])
    print(f"\nUpdated DEFAULTS in {path}")